# 2. Final evaluation on PlantVillage

Use this notebook after model selection. It evaluates one selected checkpoint on the locked PlantVillage test split. It does not train a model.

The test set has 10,709 images and 38 classes. Use the validation set to choose the model. Use the test set once for the final score.


## Attach the inputs

In Kaggle, attach the PlantVillage dataset and the selected model checkpoint. The default checkpoint path is the EfficientNet-B0 dataset path. Change `CHECKPOINT_PATH` if you select another model.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

# Confirm that model selection is complete before opening the locked test split.
CONFIRM_FINAL_MODEL_SELECTION = False
SELECTED_ARCHITECTURE = "efficientnet_b0"  # smallcnn | resnet18 | efficientnet_b0
CHECKPOINT_PATH = Path("/kaggle/input/datasets/shubhamkumar12864/efficientnet/efficientnet_b0_best.pt")
PV_ROOT = Path("/kaggle/input/datasets/shubhamkumar12864/plantvillage")
OUTPUT_DIR = Path("/kaggle/working/final_evaluation_plantvillage")
IMAGE_SIZE = 224
BATCH_SIZE = 32 if torch.cuda.device_count() >= 2 else 16
NUM_WORKERS = 0
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if not CONFIRM_FINAL_MODEL_SELECTION:
    raise RuntimeError("Select the model using validation results. Then set CONFIRM_FINAL_MODEL_SELECTION=True.")
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Model checkpoint not found: {CHECKPOINT_PATH}. Attach the checkpoint dataset in Kaggle Add Input.")
if not (PV_ROOT / "data/manifests/plantvillage_test.csv").is_file():
    raise FileNotFoundError(f"PlantVillage test manifest not found under {PV_ROOT}. Attach the PlantVillage dataset.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
classes = checkpoint.get("classes")
if not classes:
    raise ValueError("Checkpoint must contain its ordered classes list.")


## Load the selected model

The model name and the checkpoint must match. Keep the saved image normalization.


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        blocks, in_channels = [], 3
        for out_channels in (32, 64, 128, 256):
            blocks += [nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
                       nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            in_channels = out_channels
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(.30), nn.Linear(256, n_classes))
    def forward(self, x):
        return self.head(self.features(x))

def build_model(name, n_classes):
    if name == "smallcnn":
        return SmallCNN(n_classes)
    if name == "resnet18":
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, n_classes)
        return model
    if name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, n_classes)
        return model
    raise ValueError(f"Unknown architecture: {name}")

if checkpoint.get("architecture") and checkpoint["architecture"] != SELECTED_ARCHITECTURE:
    raise ValueError("Selected architecture does not match checkpoint metadata.")
state = checkpoint.get("model_state_dict", checkpoint.get("state_dict"))
if state is None:
    raise ValueError("Checkpoint has no model weights.")
model = build_model(SELECTED_ARCHITECTURE, len(classes))
model.load_state_dict(state)
model.to(DEVICE).eval()
if DEVICE.type == "cuda":
    model = nn.DataParallel(model, device_ids=list(range(min(torch.cuda.device_count(), 2))))
mean = tuple(checkpoint.get("normalization_mean", [.5, .5, .5]))
std = tuple(checkpoint.get("normalization_std", [.5, .5, .5]))
eval_transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize(mean, std)])
label_to_index = {label: i for i, label in enumerate(classes)}
print(f"Architecture: {SELECTED_ARCHITECTURE} | classes: {len(classes)} | device: {DEVICE}")


## Evaluate the locked test split

This cell reports accuracy, balanced accuracy, macro precision, macro recall, and macro-F1. It also saves a class report and confusion matrix.


In [ ]:
class ImageRows(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            return eval_transform(image.convert("RGB"))

def predict(loader):
    all_predictions = []
    with torch.inference_mode():
        for images in tqdm(loader, desc="Predict PlantVillage test images"):
            images = images.to(DEVICE, non_blocking=True)
            if DEVICE.type == "cuda":
                images = images.contiguous(memory_format=torch.channels_last)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                logits = model(images)
            all_predictions.extend(logits.argmax(1).cpu().numpy().tolist())
    return np.asarray(all_predictions, dtype=np.int64)

manifest = pd.read_csv(PV_ROOT / "data/manifests/plantvillage_test.csv")
image_root = PV_ROOT / "data/raw/PlantVillage/test"
paths = [image_root / rel for rel in manifest["relative_path"].astype(str)]
missing = [str(path) for path in paths if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing {len(missing)} test images. First missing: {missing[0]}")
y_true = np.asarray([label_to_index[label] for label in manifest["label"].astype(str)], dtype=np.int64)
loader = DataLoader(ImageRows(paths), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
y_pred = predict(loader)
metrics = {
    "dataset": "PlantVillage_test", "images": int(len(y_true)), "classes": int(len(classes)),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    "macro_precision": float(precision_score(y_true, y_pred, labels=np.arange(len(classes)), average="macro", zero_division=0)),
    "macro_recall": float(recall_score(y_true, y_pred, labels=np.arange(len(classes)), average="macro", zero_division=0)),
    "macro_f1": float(f1_score(y_true, y_pred, labels=np.arange(len(classes)), average="macro", zero_division=0)),
}
print(json.dumps(metrics, indent=2))
report = classification_report(y_true, y_pred, labels=np.arange(len(classes)), target_names=classes, output_dict=True, zero_division=0)
pd.DataFrame(report).T.to_csv(OUTPUT_DIR / "plantvillage_test_classification_report.csv")
cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(classes)))
pd.DataFrame(cm, index=classes, columns=classes).to_csv(OUTPUT_DIR / "plantvillage_test_confusion_matrix.csv")
(OUTPUT_DIR / "plantvillage_test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("Saved files:", [path.name for path in OUTPUT_DIR.iterdir()])


## Download the results

Download the files in `/kaggle/working/final_evaluation_plantvillage/` from the Kaggle Output panel. The files contain PlantVillage test metrics, a class report, and a confusion matrix. Keep the test split fixed.
